# 1024+ Dimension Codebase Analysis with HuggingFace Models

This notebook demonstrates advanced codebase analysis using high-dimensional embedding models from HuggingFace. It automatically downloads models if not available locally and generates embeddings for comprehensive code analysis.

## Features:
- ✅ Automatic model downloading and caching
- ✅ Multiple 1024+ dimension embedding models
- ✅ Comprehensive codebase analysis
- ✅ Model performance comparison
- ✅ Visualization and clustering
- ✅ Results saving and export

## Target Models:
1. **BAAI/bge-large-en-v1.5** (1024 dims) - Best overall performance
2. **thenlper/gte-large** (1024 dims) - Excellent speed and accuracy
3. **intfloat/e5-large-v2** (1024 dims) - Fast and reliable
4. **BAAI/bge-m3** (1024 dims) - Multi-lingual support

## 1. Install Required Dependencies

First, let's install all necessary packages for codebase analysis and embedding generation.

In [ ]:
# Install required packages
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to install {package}: {e}")

# Required packages for codebase analysis
required_packages = [
    "torch",
    "transformers",
    "sentence-transformers",
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "umap-learn",
    "plotly",
    "tqdm",
    "huggingface-hub"
]

print("🔧 Installing required dependencies...")
for package in required_packages:
    install_package(package)

print("\n🎉 All dependencies installed successfully!")

## 2. Import Libraries and Setup

Import all necessary libraries and configure the environment for optimal performance.

In [ ]:
# Core libraries
import os
import sys
import json
import time
import logging
import warnings
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Any
from datetime import datetime

# Data processing
import numpy as np
import pandas as pd

# Machine learning and embeddings
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Progress bars
from tqdm.auto import tqdm

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configure matplotlib for better plots
plt.style.use('default')
sns.set_palette("husl")

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✅ All libraries imported successfully!")
print(f"🐍 Python version: {sys.version}")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"📊 NumPy version: {np.__version__}")
print(f"🐼 Pandas version: {pd.__version__}")

## 3. Define High-Dimensional Model Configuration

Configure the most effective 1024+ dimension embedding models from HuggingFace for codebase analysis.

In [ ]:
# Configuration for high-dimensional embedding models
EMBEDDING_MODELS_1024 = {
    "BAAI/bge-large-en-v1.5": {
        "dimensions": 1024,
        "trust_remote_code": True,
        "description": "🥇 BEST OVERALL - Top MTEB performance, fast inference, excellent for code",
        "specialty": "English text and code",
        "performance_tier": "S-Tier"
    },
    "thenlper/gte-large": {
        "dimensions": 1024,
        "trust_remote_code": False,
        "description": "🥈 EXCELLENT SPEED - Very fast, great accuracy, multilingual support",
        "specialty": "General purpose with speed optimization",
        "performance_tier": "A-Tier"
    },
    "intfloat/e5-large-v2": {
        "dimensions": 1024,
        "trust_remote_code": False,
        "description": "⚡ FAST & RELIABLE - Excellent speed/accuracy balance, stable performance",
        "specialty": "Balanced performance",
        "performance_tier": "A-Tier"
    },
    "BAAI/bge-m3": {
        "dimensions": 1024,
        "trust_remote_code": True,
        "description": "🌍 MULTILINGUAL - Multi-lingual, multi-functionality, multi-granularity",
        "specialty": "Multilingual code and text",
        "performance_tier": "A-Tier"
    },
    "sentence-transformers/all-roberta-large-v1": {
        "dimensions": 1024,
        "trust_remote_code": False,
        "description": "🔬 RESEARCH GRADE - RoBERTa-based, excellent for semantic analysis",
        "specialty": "Semantic understanding",
        "performance_tier": "B-Tier"
    }
}

# Additional high-dimension models for comparison
ADDITIONAL_MODELS = {
    "intfloat/multilingual-e5-large": {
        "dimensions": 1024,
        "trust_remote_code": False,
        "description": "🌐 100+ LANGUAGES - Best multilingual model for diverse codebases",
        "specialty": "Multilingual code analysis",
        "performance_tier": "A-Tier"
    },
    "intfloat/e5-mistral-7b-instruct": {
        "dimensions": 4096,
        "trust_remote_code": False,
        "description": "🚀 ULTRA HIGH-DIM - 7B parameter model with 4096 dimensions",
        "specialty": "Maximum representation power",
        "performance_tier": "S-Tier (if resources allow)"
    }
}

# Model cache directory
CACHE_DIR = Path("./model_cache")
CACHE_DIR.mkdir(exist_ok=True)

print("📋 Configured High-Dimensional Models:")
print("=" * 60)
for model_name, config in EMBEDDING_MODELS_1024.items():
    print(f"🔹 {model_name}")
    print(f"   📐 Dimensions: {config['dimensions']}")
    print(f"   🎯 Tier: {config['performance_tier']}")
    print(f"   📝 {config['description']}")
    print()

print(f"💾 Cache directory: {CACHE_DIR.absolute()}")
print(f"🖥️ Target device: {device}")

## 4. Download and Load Embedding Models

Implement smart model downloading with automatic caching and fallback mechanisms.

In [ ]:
class AdvancedEmbeddingManager:
    """Advanced embedding model manager with automatic downloading and caching"""
    
    def __init__(self, cache_dir: Path = CACHE_DIR):
        self.cache_dir = cache_dir
        self.loaded_models = {}
        self.model_info = {}
        
    def get_model_cache_path(self, model_name: str) -> Path:
        """Get cache path for a specific model"""
        safe_name = model_name.replace('/', '_').replace(':', '_')
        return self.cache_dir / safe_name
    
    def is_model_cached(self, model_name: str) -> bool:
        """Check if model is already cached locally"""
        cache_path = self.get_model_cache_path(model_name)
        if not cache_path.exists():
            return False
            
        # Check for essential files
        essential_files = [
            "config_sentence_transformers.json",
            "modules.json"
        ]
        
        model_files = ["pytorch_model.bin", "model.safetensors"]
        tokenizer_files = ["tokenizer.json", "vocab.txt", "tokenizer_config.json"]
        
        # Must have essential files
        has_essential = all((cache_path / f).exists() for f in essential_files)
        
        # Must have at least one model file
        has_model = any((cache_path / f).exists() for f in model_files)
        
        # Must have at least one tokenizer file
        has_tokenizer = any((cache_path / f).exists() for f in tokenizer_files)
        
        return has_essential and has_model and has_tokenizer
    
    def download_and_cache_model(self, model_name: str, trust_remote_code: bool = False) -> bool:
        """Download and cache a model if not already cached"""
        try:
            cache_path = self.get_model_cache_path(model_name)
            
            if self.is_model_cached(model_name):
                logger.info(f"✅ Model {model_name} already cached at {cache_path}")
                return True
            
            logger.info(f"⬇️ Downloading model {model_name}...")
            start_time = time.time()
            
            # Download model using SentenceTransformer
            model = SentenceTransformer(
                model_name,
                device='cpu',  # Load to CPU first to avoid memory issues
                trust_remote_code=trust_remote_code,
                cache_folder=str(cache_path.parent)
            )
            
            # Save to our cache directory
            model.save(str(cache_path))
            
            download_time = time.time() - start_time
            logger.info(f"✅ Model {model_name} downloaded and cached in {download_time:.2f}s")
            
            # Verify caching
            if self.is_model_cached(model_name):
                logger.info(f"✅ Cache verification successful for {model_name}")
                return True
            else:
                logger.warning(f"⚠️ Cache verification failed for {model_name}")
                return False
                
        except Exception as e:
            logger.error(f"❌ Failed to download {model_name}: {e}")
            return False
    
    def load_model(self, model_name: str, trust_remote_code: bool = False) -> Optional[SentenceTransformer]:
        """Load a model from cache or download if necessary"""
        try:
            if model_name in self.loaded_models:
                logger.info(f"🔄 Reusing cached model instance: {model_name}")
                return self.loaded_models[model_name]
            
            # Ensure model is downloaded and cached
            if not self.download_and_cache_model(model_name, trust_remote_code):
                return None
            
            # Load from cache
            cache_path = self.get_model_cache_path(model_name)
            logger.info(f"📂 Loading model from cache: {cache_path}")
            
            model = SentenceTransformer(
                str(cache_path),
                device=str(device),
                trust_remote_code=trust_remote_code
            )
            
            # Store model info
            self.model_info[model_name] = {
                'dimensions': model.get_sentence_embedding_dimension(),
                'max_seq_length': getattr(model, 'max_seq_length', 'Unknown'),
                'device': str(model.device),
                'cache_path': str(cache_path)
            }
            
            self.loaded_models[model_name] = model
            logger.info(f"✅ Model {model_name} loaded successfully")
            
            return model
            
        except Exception as e:
            logger.error(f"❌ Failed to load model {model_name}: {e}")
            return None
    
    def get_model_info(self, model_name: str) -> Dict[str, Any]:
        """Get information about a loaded model"""
        return self.model_info.get(model_name, {})
    
    def list_cached_models(self) -> List[str]:
        """List all cached models"""
        cached = []
        for model_name in EMBEDDING_MODELS_1024.keys():
            if self.is_model_cached(model_name):
                cached.append(model_name)
        return cached
    
    def clear_cache(self, model_name: Optional[str] = None):
        """Clear model cache"""
        if model_name:
            cache_path = self.get_model_cache_path(model_name)
            if cache_path.exists():
                import shutil
                shutil.rmtree(cache_path)
                logger.info(f"🗑️ Cleared cache for {model_name}")
        else:
            import shutil
            if self.cache_dir.exists():
                shutil.rmtree(self.cache_dir)
                self.cache_dir.mkdir(exist_ok=True)
                logger.info("🗑️ Cleared all model caches")

# Initialize the embedding manager
embedding_manager = AdvancedEmbeddingManager()

print("🚀 Advanced Embedding Manager initialized!")
print(f"📁 Cache directory: {embedding_manager.cache_dir}")
print(f"💾 Currently cached models: {len(embedding_manager.list_cached_models())}")

## 5. Load Target Models

Download and load the most effective 1024-dimension models for codebase analysis.

In [ ]:
# Load the most effective 1024-dimension models
target_models = [
    "BAAI/bge-large-en-v1.5",  # Best overall performance
    "thenlper/gte-large",      # Excellent speed
    "intfloat/e5-large-v2",    # Fast and reliable
    "BAAI/bge-m3"              # Multilingual support
]

loaded_models = {}
model_performance = {}

print("🔄 Loading target embedding models...")
print("=" * 60)

for model_name in target_models:
    print(f"\n📦 Processing: {model_name}")
    
    config = EMBEDDING_MODELS_1024[model_name]
    trust_remote_code = config.get('trust_remote_code', False)
    
    # Check if already cached
    if embedding_manager.is_model_cached(model_name):
        print(f"   ✅ Found in cache")
    else:
        print(f"   ⬇️ Downloading (this may take a few minutes)...")
    
    start_time = time.time()
    
    # Load the model
    model = embedding_manager.load_model(model_name, trust_remote_code)
    
    if model is not None:
        load_time = time.time() - start_time
        loaded_models[model_name] = model
        
        # Get model information
        model_info = embedding_manager.get_model_info(model_name)
        model_performance[model_name] = {
            'load_time': load_time,
            'dimensions': model_info.get('dimensions', 'Unknown'),
            'max_seq_length': model_info.get('max_seq_length', 'Unknown'),
            'device': model_info.get('device', 'Unknown'),
            'description': config['description'],
            'performance_tier': config['performance_tier']
        }
        
        print(f"   ✅ Loaded successfully in {load_time:.2f}s")
        print(f"   📐 Dimensions: {model_info.get('dimensions')}")
        print(f"   🎯 Tier: {config['performance_tier']}")
    else:
        print(f"   ❌ Failed to load")

print(f"\n🎉 Successfully loaded {len(loaded_models)}/{len(target_models)} models!")

# Display summary
if loaded_models:
    print("\n📊 Model Loading Summary:")
    print("=" * 80)
    
    df_summary = pd.DataFrame(model_performance).T
    df_summary['load_time'] = df_summary['load_time'].round(2)
    
    print(df_summary[['dimensions', 'performance_tier', 'load_time', 'device']].to_string())
    
    print(f"\n💾 Total cache size: {sum(p.stat().st_size for p in embedding_manager.cache_dir.rglob('*') if p.is_file()) / (1024**3):.2f} GB")
else:
    print("❌ No models were loaded successfully. Please check your internet connection and try again.")

## 6. Codebase Analysis Functions

Implement comprehensive functions to analyze codebases with intelligent file processing.

In [ ]:
class CodebaseAnalyzer:
    """Advanced codebase analyzer with intelligent code processing"""
    
    def __init__(self, models: Dict[str, SentenceTransformer]):
        self.models = models
        self.code_extensions = {
            '.py', '.js', '.ts', '.java', '.cpp', '.c', '.cs', '.php',
            '.rb', '.go', '.rs', '.swift', '.kt', '.scala', '.r', '.m',
            '.h', '.hpp', '.jsx', '.tsx', '.vue', '.sql', '.sh', '.bat'
        }
        self.ignore_patterns = {
            '__pycache__', '.git', '.svn', 'node_modules', '.vscode',
            '.idea', 'dist', 'build', '.pytest_cache', 'venv', 'env'
        }
    
    def is_code_file(self, file_path: Path) -> bool:
        """Check if file is a code file"""
        return file_path.suffix.lower() in self.code_extensions
    
    def should_ignore_path(self, path: Path) -> bool:
        """Check if path should be ignored"""
        return any(pattern in str(path) for pattern in self.ignore_patterns)
    
    def read_code_file(self, file_path: Path) -> Optional[str]:
        """Read code file with encoding detection"""
        try:
            # Try UTF-8 first
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
        except UnicodeDecodeError:
            try:
                # Try with different encodings
                for encoding in ['utf-8', 'latin-1', 'cp1252']:
                    try:
                        with open(file_path, 'r', encoding=encoding) as f:
                            return f.read()
                    except:
                        continue
            except Exception as e:
                logger.warning(f"Could not read {file_path}: {e}")
                return None
        except Exception as e:
            logger.warning(f"Error reading {file_path}: {e}")
            return None
    
    def extract_code_snippets(self, content: str, file_path: Path, max_length: int = 500) -> List[Dict[str, str]]:
        """Extract meaningful code snippets from file content"""
        snippets = []
        
        # Split by lines
        lines = content.split('\n')
        current_snippet = []
        current_length = 0
        
        for i, line in enumerate(lines):
            # Skip empty lines and comments for snippet boundaries
            stripped = line.strip()
            if not stripped or stripped.startswith('#') or stripped.startswith('//'):
                if current_snippet and current_length > 50:  # Min snippet length
                    snippets.append({
                        'content': '\n'.join(current_snippet),
                        'file_path': str(file_path),
                        'start_line': i - len(current_snippet) + 1,
                        'end_line': i,
                        'language': file_path.suffix[1:] if file_path.suffix else 'text'
                    })
                    current_snippet = []
                    current_length = 0
                continue
            
            current_snippet.append(line)
            current_length += len(line)
            
            # Split if snippet gets too long
            if current_length > max_length:
                snippets.append({
                    'content': '\n'.join(current_snippet),
                    'file_path': str(file_path),
                    'start_line': i - len(current_snippet) + 1,
                    'end_line': i,
                    'language': file_path.suffix[1:] if file_path.suffix else 'text'
                })
                current_snippet = []
                current_length = 0
        
        # Add final snippet if exists
        if current_snippet and current_length > 50:
            snippets.append({
                'content': '\n'.join(current_snippet),
                'file_path': str(file_path),
                'start_line': len(lines) - len(current_snippet) + 1,
                'end_line': len(lines),
                'language': file_path.suffix[1:] if file_path.suffix else 'text'
            })
        
        return snippets
    
    def scan_codebase(self, root_path: str, max_files: int = 100) -> List[Dict[str, str]]:
        """Scan codebase and extract code snippets"""
        root = Path(root_path)
        all_snippets = []
        files_processed = 0
        
        print(f"🔍 Scanning codebase: {root}")
        print(f"📁 Looking for files with extensions: {', '.join(sorted(self.code_extensions))}")
        
        pbar = tqdm(desc="Processing files", unit="files")
        
        for file_path in root.rglob('*'):
            if files_processed >= max_files:
                break
                
            if not file_path.is_file() or not self.is_code_file(file_path):
                continue
                
            if self.should_ignore_path(file_path):
                continue
            
            content = self.read_code_file(file_path)
            if content:
                snippets = self.extract_code_snippets(content, file_path)
                all_snippets.extend(snippets)
                files_processed += 1
                pbar.update(1)
                pbar.set_postfix({"snippets": len(all_snippets), "files": files_processed})
        
        pbar.close()
        
        print(f"✅ Processed {files_processed} files")
        print(f"📄 Extracted {len(all_snippets)} code snippets")
        
        return all_snippets
    
    def generate_embeddings(self, snippets: List[Dict[str, str]], model_name: str, batch_size: int = 32) -> np.ndarray:
        """Generate embeddings for code snippets using specified model"""
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} not loaded")
        
        model = self.models[model_name]
        texts = [snippet['content'] for snippet in snippets]
        
        print(f"🧮 Generating embeddings with {model_name}...")
        print(f"📊 Processing {len(texts)} code snippets")
        
        # Generate embeddings in batches
        embeddings = []
        for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
            batch = texts[i:i + batch_size]
            batch_embeddings = model.encode(batch, convert_to_tensor=False, show_progress_bar=False)
            embeddings.extend(batch_embeddings)
        
        embeddings_array = np.array(embeddings)
        print(f"✅ Generated embeddings shape: {embeddings_array.shape}")
        
        return embeddings_array
    
    def analyze_codebase_with_models(self, root_path: str, max_files: int = 50) -> Dict[str, Any]:
        """Analyze codebase with all loaded models"""
        # Scan codebase
        snippets = self.scan_codebase(root_path, max_files)
        
        if not snippets:
            print("❌ No code snippets found")
            return {}
        
        results = {
            'snippets': snippets,
            'embeddings': {},
            'analysis_timestamp': datetime.now().isoformat(),
            'root_path': root_path,
            'total_snippets': len(snippets)
        }
        
        # Generate embeddings with each model
        for model_name in self.models.keys():
            try:
                embeddings = self.generate_embeddings(snippets, model_name)
                results['embeddings'][model_name] = embeddings
                print(f"✅ {model_name}: {embeddings.shape}")
            except Exception as e:
                print(f"❌ {model_name}: {e}")
        
        return results

# Initialize analyzer with loaded models
if loaded_models:
    analyzer = CodebaseAnalyzer(loaded_models)
    print("🔬 Codebase Analyzer initialized!")
    print(f"🤖 Available models: {list(loaded_models.keys())}")
else:
    print("❌ No models available for analysis")

## 7. Analyze Current Codebase

Let's analyze the current uphire_v2 codebase with our 1024-dimension models.

In [ ]:
# Analyze the current uphire_v2 codebase
current_codebase_path = "."  # Current directory (uphire_v2)

if 'analyzer' in locals() and analyzer:
    print("🚀 Starting comprehensive codebase analysis...")
    print("=" * 60)
    
    # Run analysis with all loaded models
    analysis_results = analyzer.analyze_codebase_with_models(
        root_path=current_codebase_path,
        max_files=30  # Limit for demonstration
    )
    
    if analysis_results:
        print(f"\n📈 Analysis Results Summary:")
        print(f"📁 Root path: {analysis_results['root_path']}")
        print(f"📄 Total snippets: {analysis_results['total_snippets']}")
        print(f"🤖 Models used: {list(analysis_results['embeddings'].keys())}")
        print(f"⏰ Analysis time: {analysis_results['analysis_timestamp']}")
        
        # Show sample snippets
        if analysis_results['snippets']:
            print(f"\n📋 Sample Code Snippets:")
            print("-" * 40)
            
            for i, snippet in enumerate(analysis_results['snippets'][:3]):
                print(f"\nSnippet {i+1}:")
                print(f"  📁 File: {snippet['file_path']}")
                print(f"  📝 Language: {snippet['language']}")
                print(f"  📏 Lines: {snippet['start_line']}-{snippet['end_line']}")
                print(f"  💾 Content preview: {snippet['content'][:100]}...")
        
        # Show embedding statistics
        print(f"\n🧮 Embedding Statistics:")
        print("-" * 40)
        
        for model_name, embeddings in analysis_results['embeddings'].items():
            print(f"\n{model_name}:")
            print(f"  📐 Shape: {embeddings.shape}")
            print(f"  📊 Mean: {embeddings.mean():.4f}")
            print(f"  📈 Std: {embeddings.std():.4f}")
            print(f"  🔢 Min: {embeddings.min():.4f}")
            print(f"  🔢 Max: {embeddings.max():.4f}")
        
        print(f"\n✅ Analysis completed successfully!")
        
        # Store results for visualization
        global codebase_analysis
        codebase_analysis = analysis_results
        
    else:
        print("❌ Analysis failed - no results generated")
        
else:
    print("❌ Analyzer not available - please run the previous cells first")

## 8. Compare Model Performance

Analyze and compare the performance of different 1024-dimension embedding models.

In [ ]:
def compare_model_performance(analysis_results: Dict[str, Any]) -> Dict[str, Any]:
    """Compare performance of different embedding models"""
    
    if not analysis_results or 'embeddings' not in analysis_results:
        print("❌ No analysis results available for comparison")
        return {}
    
    comparison_results = {
        'model_stats': {},
        'similarity_analysis': {},
        'clustering_quality': {},
        'performance_metrics': {}
    }
    
    embeddings_data = analysis_results['embeddings']
    snippets = analysis_results['snippets']
    
    print("🔍 Comparing model performance...")
    print("=" * 50)
    
    # 1. Basic statistics comparison
    print("\n📊 Model Statistics Comparison:")
    stats_data = []
    
    for model_name, embeddings in embeddings_data.items():
        stats = {
            'model': model_name.split('/')[-1],  # Short name
            'dimensions': embeddings.shape[1],
            'mean_magnitude': np.linalg.norm(embeddings, axis=1).mean(),
            'std_magnitude': np.linalg.norm(embeddings, axis=1).std(),
            'mean_value': embeddings.mean(),
            'std_value': embeddings.std(),
            'sparsity': (embeddings == 0).sum() / embeddings.size * 100
        }
        stats_data.append(stats)
        comparison_results['model_stats'][model_name] = stats
    
    df_stats = pd.DataFrame(stats_data)
    print(df_stats.round(4))
    
    # 2. Inter-model similarity analysis
    print(f"\n🔗 Inter-Model Similarity Analysis:")
    
    model_names = list(embeddings_data.keys())
    similarity_matrix = np.zeros((len(model_names), len(model_names)))
    
    for i, model1 in enumerate(model_names):
        for j, model2 in enumerate(model_names):
            if i <= j:  # Only compute upper triangle
                emb1 = embeddings_data[model1]
                emb2 = embeddings_data[model2]
                
                # Compute average cosine similarity between corresponding embeddings
                similarities = []
                for k in range(min(len(emb1), len(emb2))):
                    sim = cosine_similarity([emb1[k]], [emb2[k]])[0][0]
                    similarities.append(sim)
                
                avg_similarity = np.mean(similarities)
                similarity_matrix[i][j] = avg_similarity
                similarity_matrix[j][i] = avg_similarity
                
                if i != j:
                    print(f"  {model1.split('/')[-1]} ↔ {model2.split('/')[-1]}: {avg_similarity:.4f}")
    
    comparison_results['similarity_analysis']['matrix'] = similarity_matrix
    comparison_results['similarity_analysis']['model_names'] = [m.split('/')[-1] for m in model_names]
    
    # 3. Clustering quality assessment
    print(f"\n🎯 Clustering Quality Assessment:")
    
    from sklearn.cluster import KMeans
    from sklearn.metrics import silhouette_score
    
    for model_name, embeddings in embeddings_data.items():
        try:
            # Use K-means clustering to assess embedding quality
            n_clusters = min(5, len(embeddings) // 2)  # Adaptive cluster count
            
            if n_clusters >= 2:
                kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
                cluster_labels = kmeans.fit_predict(embeddings)
                
                # Calculate silhouette score
                silhouette_avg = silhouette_score(embeddings, cluster_labels)
                
                # Calculate inertia (within-cluster sum of squares)
                inertia = kmeans.inertia_
                
                comparison_results['clustering_quality'][model_name] = {
                    'silhouette_score': silhouette_avg,
                    'inertia': inertia,
                    'n_clusters': n_clusters
                }
                
                print(f"  {model_name.split('/')[-1]}:")
                print(f"    🎯 Silhouette Score: {silhouette_avg:.4f}")
                print(f"    📊 Inertia: {inertia:.2f}")
            
        except Exception as e:
            print(f"  ❌ {model_name.split('/')[-1]}: Clustering failed ({e})")
    
    # 4. Performance ranking
    print(f"\n🏆 Performance Ranking:")
    
    ranking_scores = {}
    
    for model_name in embeddings_data.keys():
        score = 0
        
        # Silhouette score contribution (higher is better)
        if model_name in comparison_results['clustering_quality']:
            silhouette = comparison_results['clustering_quality'][model_name]['silhouette_score']
            score += silhouette * 100  # Scale up
        
        # Magnitude consistency (lower std is better)
        stats = comparison_results['model_stats'][model_name]
        magnitude_consistency = 1 / (1 + stats['std_magnitude'])
        score += magnitude_consistency * 50
        
        # Sparsity penalty (lower sparsity is better for dense embeddings)
        sparsity_penalty = max(0, 100 - stats['sparsity']) * 0.1
        score += sparsity_penalty
        
        ranking_scores[model_name] = score
    
    # Sort by score
    sorted_models = sorted(ranking_scores.items(), key=lambda x: x[1], reverse=True)
    
    for i, (model_name, score) in enumerate(sorted_models):
        tier = ['🥇', '🥈', '🥉', '🏅', '⭐'][min(i, 4)]
        print(f"  {tier} {model_name.split('/')[-1]}: {score:.2f}")
    
    comparison_results['performance_metrics']['ranking'] = sorted_models
    
    return comparison_results

# Run comparison if analysis results are available
if 'codebase_analysis' in locals() and codebase_analysis:
    performance_comparison = compare_model_performance(codebase_analysis)
    print(f"\n✅ Model performance comparison completed!")
else:
    print("❌ No codebase analysis results available for comparison")

## 9. Visualize Embeddings

Create comprehensive visualizations of the 1024-dimension embeddings using dimensionality reduction.

In [ ]:
def create_embedding_visualizations(analysis_results: Dict[str, Any]) -> Dict[str, Any]:
    """Create comprehensive visualizations of embeddings"""
    
    if not analysis_results or 'embeddings' not in analysis_results:
        print("❌ No analysis results available for visualization")
        return {}
    
    embeddings_data = analysis_results['embeddings']
    snippets = analysis_results['snippets']
    
    print("🎨 Creating embedding visualizations...")
    print("=" * 50)
    
    # Create subplots for multiple visualizations
    n_models = len(embeddings_data)
    fig = make_subplots(
        rows=2, cols=n_models,
        subplot_titles=[f"{name.split('/')[-1]}" for name in embeddings_data.keys()] * 2,
        specs=[[{"secondary_y": False}] * n_models] * 2,
        row_titles=["t-SNE Visualization", "PCA Visualization"]
    )
    
    colors = px.colors.qualitative.Set3
    viz_results = {}
    
    for i, (model_name, embeddings) in enumerate(embeddings_data.items()):
        print(f"📊 Processing {model_name}...")
        
        # Limit embeddings for visualization performance
        max_points = min(200, len(embeddings))
        sample_embeddings = embeddings[:max_points]
        sample_snippets = snippets[:max_points]
        
        # Create file type labels for coloring
        file_types = [snippet['language'] for snippet in sample_snippets]
        unique_types = list(set(file_types))
        type_colors = {t: colors[j % len(colors)] for j, t in enumerate(unique_types)}
        
        try:
            # 1. t-SNE Visualization
            print(f"  🔄 Computing t-SNE...")
            tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, max_points-1))
            tsne_results = tsne.fit_transform(sample_embeddings)
            
            # Add t-SNE plot
            for file_type in unique_types:
                mask = np.array(file_types) == file_type
                fig.add_scatter(
                    x=tsne_results[mask, 0],
                    y=tsne_results[mask, 1],
                    mode='markers',
                    name=f"{file_type}",
                    marker=dict(color=type_colors[file_type], size=6),
                    row=1, col=i+1,
                    showlegend=(i == 0)  # Only show legend for first plot
                )
            
            # 2. PCA Visualization
            print(f"  🔄 Computing PCA...")
            pca = PCA(n_components=2, random_state=42)
            pca_results = pca.fit_transform(sample_embeddings)
            
            # Add PCA plot
            for file_type in unique_types:
                mask = np.array(file_types) == file_type
                fig.add_scatter(
                    x=pca_results[mask, 0],
                    y=pca_results[mask, 1],
                    mode='markers',
                    name=f"{file_type}",
                    marker=dict(color=type_colors[file_type], size=6),
                    row=2, col=i+1,
                    showlegend=False
                )
            
            # Store results
            viz_results[model_name] = {
                'tsne_results': tsne_results,
                'pca_results': pca_results,
                'file_types': file_types,
                'pca_explained_variance': pca.explained_variance_ratio_
            }
            
        except Exception as e:
            print(f"  ❌ Visualization failed for {model_name}: {e}")
    
    # Update layout
    fig.update_layout(
        height=800,
        title_text="1024-Dimension Embedding Visualizations",
        title_x=0.5,
        font=dict(size=10)
    )
    
    # Show the plot
    fig.show()
    
    # Create additional comparison plots
    print(f"\n📈 Creating comparison plots...")
    
    # 1. Model similarity heatmap
    if 'performance_comparison' in locals():
        fig_heatmap = go.Figure(data=go.Heatmap(
            z=performance_comparison['similarity_analysis']['matrix'],
            x=performance_comparison['similarity_analysis']['model_names'],
            y=performance_comparison['similarity_analysis']['model_names'],
            colorscale='Viridis',
            text=performance_comparison['similarity_analysis']['matrix'],
            texttemplate="%{text:.3f}",
            textfont={"size": 10}
        ))
        
        fig_heatmap.update_layout(
            title="Inter-Model Similarity Matrix",
            xaxis_title="Models",
            yaxis_title="Models",
            width=600,
            height=500
        )
        fig_heatmap.show()
    
    # 2. Performance comparison bar chart
    if 'performance_comparison' in locals() and 'ranking' in performance_comparison['performance_metrics']:
        ranking_data = performance_comparison['performance_metrics']['ranking']
        
        fig_bar = go.Figure(data=[
            go.Bar(
                x=[name.split('/')[-1] for name, score in ranking_data],
                y=[score for name, score in ranking_data],
                marker_color=['gold', 'silver', '#CD7F32', 'lightblue', 'lightgray'][:len(ranking_data)]
            )
        ])
        
        fig_bar.update_layout(
            title="Model Performance Ranking",
            xaxis_title="Models",
            yaxis_title="Performance Score",
            width=800,
            height=400
        )
        fig_bar.show()
    
    return viz_results

# Create visualizations if analysis results are available
if 'codebase_analysis' in locals() and codebase_analysis:
    try:
        visualization_results = create_embedding_visualizations(codebase_analysis)
        print(f"\n✅ Embedding visualizations created successfully!")
    except Exception as e:
        print(f"❌ Visualization failed: {e}")
        print("📝 Note: Install umap-learn if t-SNE/PCA visualizations are not working properly")
else:
    print("❌ No codebase analysis results available for visualization")

## 10. Save Results and Export

Save analysis results, embeddings, and reports for future use.

In [ ]:
def save_analysis_results(
    analysis_results: Dict[str, Any],
    performance_comparison: Dict[str, Any] = None,
    output_dir: str = "./analysis_output"
) -> Dict[str, str]:
    """Save all analysis results to files"""
    
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    saved_files = {}
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    print(f"💾 Saving analysis results to: {output_path}")
    print("=" * 50)
    
    try:
        # 1. Save embeddings as numpy arrays
        embeddings_dir = output_path / "embeddings"
        embeddings_dir.mkdir(exist_ok=True)
        
        for model_name, embeddings in analysis_results['embeddings'].items():
            safe_name = model_name.replace('/', '_').replace(':', '_')
            embedding_file = embeddings_dir / f"{safe_name}_{timestamp}.npy"
            np.save(embedding_file, embeddings)
            saved_files[f"embeddings_{safe_name}"] = str(embedding_file)
            print(f"✅ Saved embeddings: {embedding_file.name}")
        
        # 2. Save code snippets as JSON
        snippets_file = output_path / f"code_snippets_{timestamp}.json"
        
        # Convert snippets to serializable format
        snippets_data = {
            'metadata': {
                'timestamp': analysis_results['analysis_timestamp'],
                'root_path': analysis_results['root_path'],
                'total_snippets': analysis_results['total_snippets']
            },
            'snippets': analysis_results['snippets']
        }
        
        with open(snippets_file, 'w', encoding='utf-8') as f:
            json.dump(snippets_data, f, indent=2, ensure_ascii=False)
        
        saved_files['snippets'] = str(snippets_file)
        print(f"✅ Saved snippets: {snippets_file.name}")
        
        # 3. Save performance comparison results
        if performance_comparison:
            comparison_file = output_path / f"performance_comparison_{timestamp}.json"
            
            # Convert numpy arrays to lists for JSON serialization
            comparison_serializable = {}
            for key, value in performance_comparison.items():
                if key == 'similarity_analysis' and 'matrix' in value:
                    comparison_serializable[key] = {
                        'matrix': value['matrix'].tolist(),
                        'model_names': value['model_names']
                    }
                else:
                    comparison_serializable[key] = value
            
            with open(comparison_file, 'w', encoding='utf-8') as f:
                json.dump(comparison_serializable, f, indent=2, ensure_ascii=False)
            
            saved_files['performance_comparison'] = str(comparison_file)
            print(f"✅ Saved comparison: {comparison_file.name}")
        
        # 4. Create comprehensive analysis report
        report_file = output_path / f"analysis_report_{timestamp}.md"
        
        with open(report_file, 'w', encoding='utf-8') as f:
            f.write(f"# 1024-Dimension Codebase Analysis Report\\n\\n")
            f.write(f"**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\\n")
            f.write(f"**Codebase**: {analysis_results['root_path']}\\n")
            f.write(f"**Total Snippets**: {analysis_results['total_snippets']}\\n\\n")
            
            f.write(f"## Models Analyzed\\n\\n")
            for model_name, embeddings in analysis_results['embeddings'].items():
                f.write(f"- **{model_name}**: {embeddings.shape[0]} embeddings × {embeddings.shape[1]} dimensions\\n")
            
            f.write(f"\\n## File Types Processed\\n\\n")
            file_types = {}
            for snippet in analysis_results['snippets']:
                lang = snippet['language']
                file_types[lang] = file_types.get(lang, 0) + 1
            
            for lang, count in sorted(file_types.items()):
                f.write(f"- **{lang}**: {count} snippets\\n")
            
            f.write(f"\\n## Performance Ranking\\n\\n")
            if performance_comparison and 'ranking' in performance_comparison['performance_metrics']:
                for i, (model_name, score) in enumerate(performance_comparison['performance_metrics']['ranking']):
                    medal = ['🥇', '🥈', '🥉', '🏅', '⭐'][min(i, 4)]
                    f.write(f"{i+1}. {medal} **{model_name.split('/')[-1]}**: {score:.2f}\\n")
            
            f.write(f"\\n## Files Generated\\n\\n")
            for desc, path in saved_files.items():
                f.write(f"- **{desc}**: `{Path(path).name}`\\n")
        
        saved_files['report'] = str(report_file)
        print(f"✅ Saved report: {report_file.name}")
        
        # 5. Create summary CSV for easy analysis
        summary_file = output_path / f"embedding_summary_{timestamp}.csv"
        
        summary_data = []
        for snippet in analysis_results['snippets']:
            row = {
                'file_path': snippet['file_path'],
                'language': snippet['language'],
                'start_line': snippet['start_line'],
                'end_line': snippet['end_line'],
                'content_length': len(snippet['content']),
                'content_preview': snippet['content'][:100].replace('\\n', ' ')
            }
            
            # Add embedding statistics for each model
            snippet_idx = analysis_results['snippets'].index(snippet)
            for model_name, embeddings in analysis_results['embeddings'].items():
                if snippet_idx < len(embeddings):
                    embedding = embeddings[snippet_idx]
                    safe_name = model_name.split('/')[-1]
                    row[f'{safe_name}_magnitude'] = np.linalg.norm(embedding)
                    row[f'{safe_name}_mean'] = embedding.mean()
                    row[f'{safe_name}_std'] = embedding.std()
            
            summary_data.append(row)
        
        df_summary = pd.DataFrame(summary_data)
        df_summary.to_csv(summary_file, index=False)
        saved_files['summary_csv'] = str(summary_file)
        print(f"✅ Saved summary: {summary_file.name}")
        
        print(f"\\n🎉 All results saved successfully!")
        print(f"📁 Output directory: {output_path.absolute()}")
        print(f"📊 Total files created: {len(saved_files)}")
        
        return saved_files
        
    except Exception as e:
        print(f"❌ Error saving results: {e}")
        return {}

# Save results if available
results_saved = {}

if 'codebase_analysis' in locals() and codebase_analysis:
    comparison_data = performance_comparison if 'performance_comparison' in locals() else None
    
    results_saved = save_analysis_results(
        analysis_results=codebase_analysis,
        performance_comparison=comparison_data,
        output_dir="./codebase_analysis_output"
    )
    
    if results_saved:
        print(f"\\n📋 Saved Files Summary:")
        print("-" * 30)
        for desc, path in results_saved.items():
            print(f"📄 {desc}: {Path(path).name}")
    
else:
    print("❌ No analysis results available to save")

## 🎉 Analysis Complete!

This notebook has demonstrated a comprehensive codebase analysis using the most effective 1024-dimension embedding models from HuggingFace.

### Key Features Implemented:

✅ **Automatic Model Management**
- Smart model downloading with caching
- Fallback mechanisms for reliability
- Efficient memory management

✅ **Advanced Codebase Analysis**
- Intelligent file type detection
- Code snippet extraction
- Multi-language support

✅ **Multiple High-Performance Models**
- BAAI/bge-large-en-v1.5 (Best overall)
- thenlper/gte-large (Excellent speed)
- intfloat/e5-large-v2 (Fast & reliable)
- BAAI/bge-m3 (Multilingual)

✅ **Comprehensive Evaluation**
- Performance comparison
- Clustering quality assessment
- Inter-model similarity analysis

✅ **Rich Visualizations**
- t-SNE dimensionality reduction
- PCA analysis
- Interactive plots with Plotly

✅ **Complete Results Export**
- Embeddings as NumPy arrays
- Analysis reports in Markdown
- Summary data in CSV format
- Performance metrics in JSON

### Usage Instructions:

1. **Run all cells sequentially** to ensure proper setup
2. **Models will be automatically downloaded** if not cached
3. **Analysis targets the current codebase** by default
4. **Results are saved** to `./codebase_analysis_output/`
5. **Customize paths and parameters** as needed

### Next Steps:

- **Scale up analysis** by increasing `max_files` parameter
- **Try different codebases** by changing the `root_path`
- **Experiment with additional models** from the configuration
- **Use saved embeddings** for downstream tasks like similarity search
- **Fine-tune models** on your specific domain if needed

### Performance Notes:

- **Model caching** significantly speeds up subsequent runs
- **Batch processing** optimizes memory usage
- **GPU acceleration** available if CUDA is detected
- **File filtering** prevents processing of irrelevant files

Happy coding and embedding analysis! 🚀